In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DISCO_USECASES/

Mounted at /content/drive
/content/drive/MyDrive/DISCO_USECASES


In [2]:
########predefined functions, just run it

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.spatial.distance import jensenshannon
from scipy.optimize import linear_sum_assignment
import torch
import numpy as np
from scipy.stats import entropy

# @title
import torch
import numpy as np
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

def infer_cell_map(latent_image, model):
    """
    latent_image: torch.Tensor of shape [3, H, W], normalized to [0, 1] or [-1, 1]
    model: trained Autoencoder (only decoder is used)

    Returns:
        pred_image: torch.Tensor of shape [1, H, W], with predicted cell type index (0~24),
                    pure white points are directly assigned as 0
    """
    H, W = latent_image.shape[1], latent_image.shape[2]
    device = next(model.parameters()).device

    model.eval()
    with torch.no_grad():
        # Flatten image: [3, H, W] → [H*W, 3]
        flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3).to(device)

        # Mask: pure white pixel (assume normalized to [0, 1])
        white_mask = (flat_img == 1.0).all(dim=1)  # [H*W], bool

        # Prepare input for model only where not pure white
        infer_input = flat_img[~white_mask]  # [N_nonwhite, 3]
        pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device=device)  # [H*W]

        if infer_input.shape[0] > 0:
            logits = model.decoder(infer_input)  # [N_nonwhite, 25]
            pred_nonwhite = torch.argmax(logits, dim=1)  # [N_nonwhite]
            pred[~white_mask] = pred_nonwhite  # Assign to result

        pred_image = pred.reshape(1, H, W)  # [1, H, W]

    return pred_image


from PIL import Image
from torchvision import transforms
import torch


def load_and_recover_z3d_png(path):
    """
    Load a PNG image as RGB, interpret it as normalized z_3d embedding encoded as uint8 [0–255],
    and recover the original z_3d float values via inverse scaling.
    """
    # 1. Load RGB image as uint8 tensor
    img = Image.open(path).convert("RGB")
    arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()  # [3, H, W], uint8
    mask = (rgb > 249).all(dim=0)
    rgb[:, mask] = 255
    rgb_float = rgb.float() / 255.0  # [3, H, W], float32
    non_white_pixels = ((rgb_float != 1).any(dim=0)).sum().item()
    print(f"count for non white: {non_white_pixels}")

    return rgb_float


yticklabels = [
    "1: B",
    "2: CD4+ T cell",
    "3: CD57+ Enterocyte",
    "4: CD66+ Enterocyte",
    "5: CD7+ Immune",
    "6: CD8+ T",
    "7: Cycling TA",
    "8: DC",
    "9: Endothelial",
    "10: Enterocyte",
    "11: Goblet",
    "12: ICC",
    "13: Lymphatic",
    "14: M1 Macrophage",
    "15: M2 Macrophage",
    "16: MUC1+ Enterocyte",
    "17: NK",
    "18: Nerve",
    "19: Neuroendocrine",
    "20: Neutrophil",
    "21: Paneth",
    "22: Plasma",
    "23: Smooth muscle",
    "24: Stroma",
    "25: TA"
]

cell_type_names = {
    1:  "B", 2:  "CD4+ T", 3:  "CD57+ Ent", 4:  "CD66+ Ent", 5:  "CD7+ Imm",
    6:  "CD8+ T", 7:  "Cycling TA", 8:  "DC", 9:  "Endothelial", 10: "Enterocyte",
    11: "Goblet", 12: "ICC", 13: "Lymphatic", 14: "M1 Mac", 15: "M2 Mac",
    16: "MUC1+ Ent", 17: "NK", 18: "Nerve", 19: "Neuroendocrine", 20: "Neutrophil",
    21: "Paneth", 22: "Plasma", 23: "Smooth muscle", 24: "Stroma", 25: "TA"
}

def compute_cluster_celltype_composition(cluster_labels, cell_types, n_clusters=5, n_types=34):
    comp = np.zeros((n_clusters, n_types), dtype=np.float32)
    for i in range(len(cluster_labels)):
        c = cluster_labels[i]
        t = cell_types[i] - 1
        comp[c, t] += 1
    #comp /= (comp.sum(axis=0, keepdims=True) + 1e-8)
    return comp

def plot_cluster_map(cluster_labels, coords, title, H, W, palette=None, size=3, legend=False):
    coords = np.array(coords)
    x = coords[:, 0]
    y = coords[:, 1]
    labels = cluster_labels

    plt.figure(figsize=(10, 10))
    ax = plt.gca()

    unique_labels = np.unique(labels)

    for label in unique_labels:
        mask = labels == label
        color = palette[label] if palette else None
        ax.scatter(x[mask], y[mask], s=size, c=color, label=f"Cluster {label}", marker='s', linewidth=0)

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('white')
    plt.title(title)

    if legend:
        ax.legend(loc='upper right', fontsize=6, frameon=False)

    plt.tight_layout()
    plt.show()

def log2_fc_heatmap_from_centroids1(
    niche_clusters, tissue_values, cell_types_fixed_order, k_value, row_order=None
):
    tissue_avg = tissue_values + 1e-8
    print("tissue", tissue_avg)
    print("cluster",niche_clusters)

    smoothed = niche_clusters + tissue_avg

    smoothed_norm = smoothed / (smoothed.sum(axis=1, keepdims=True) + 1e-8)

    fc_log = np.log2(smoothed_norm / tissue_avg)

    if row_order is not None:
        fc_log = fc_log[row_order]

    if len(tissue_avg) == 25:
        df = pd.DataFrame(fc_log, columns=cell_types_fixed_order)
    else:
        df = pd.DataFrame(fc_log)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df.fillna(0, inplace=True)

    if df.isnull().values.any() or np.isinf(df.values).any():
        print("Warning: DataFrame still contains NaN or inf values!")

    plt.figure(figsize=(10, 8))

    if len(tissue_avg) == 25:
        sns.clustermap(
            df,
            vmin=-3,
            vmax=3,
            cmap='bwr',
            xticklabels=cell_types_fixed_order,
            yticklabels=[f"C{i}" for i in range(fc_log.shape[0])]
        )
    else:
        sns.clustermap(
            df,
            vmin=-3,
            vmax=3,
            cmap='bwr'
        )

    plt.title(f"Log2 FC Heatmap (k={k_value})")
    plt.xlabel("Cell Type")
    plt.ylabel("Cluster Centroid")
    plt.tight_layout()
    plt.show()


In [4]:
####metrics
!pip install lpips
def rgb_centroid_distance_score(map_true, map_gen, type_true, type_gen, visualize=True, penalty_value=2.5):
    """
    Compute centroid distance for each cell type in RGB space.

    Args:
        map_true: torch.Tensor [3, H, W] - RGB map of true image
        map_gen: torch.Tensor [3, H, W] - RGB map of generated image
        type_true: torch.Tensor [1, H, W] - cell type map of true image
        type_gen: torch.Tensor [1, H, W] - cell type map of generated image
        visualize: bool - whether to plot centroids
        penalty_value: float - penalty for missing types

    Returns:
        score: float - average similarity score
    """
    if map_true.shape[0] == 3:
        map_true = map_true.permute(1, 2, 0).cpu().numpy()
    if map_gen.shape[0] == 3:
        map_gen = map_gen.permute(1, 2, 0).cpu().numpy()

    type_true = type_true.squeeze().cpu().numpy()
    type_gen = type_gen.squeeze().cpu().numpy()

    centroids_true = {}
    centroids_gen = {}
    all_types = set(np.unique(type_true)).union(np.unique(type_gen))
    all_types.discard(0)  # ignore background

    for t in all_types:
        coords_true = np.argwhere(type_true == t)
        coords_gen = np.argwhere(type_gen == t)

        if len(coords_true) > 0:
            rgbs_true = map_true[coords_true[:, 0], coords_true[:, 1]]
            centroids_true[t] = rgbs_true.mean(axis=0)

        if len(coords_gen) > 0:
            rgbs_gen = map_gen[coords_gen[:, 0], coords_gen[:, 1]]
            centroids_gen[t] = rgbs_gen.mean(axis=0)

    distances = []
    for t in sorted(all_types):
        if t in centroids_true and t in centroids_gen:
            d = np.linalg.norm(centroids_true[t] - centroids_gen[t])
        else:
            d = penalty_value
        distances.append(d)

    if visualize:
        fig = plt.figure(figsize=(13, 8))
        ax = fig.add_subplot(111, projection='3d')

        for t in sorted(all_types):
            color = centroids_true.get(t, centroids_gen.get(t, [0.5, 0.5, 0.5]))
            if t in centroids_true:
                ax.scatter(*centroids_true[t], marker='o', color=color, s=80, label=cell_type_names[t])
            if t in centroids_gen:
                ax.scatter(*centroids_gen[t], marker='^', color=color, s=80, label=cell_type_names[t])
            if t in centroids_true and t in centroids_gen:
                ax.plot(
                    [centroids_true[t][0], centroids_gen[t][0]],
                    [centroids_true[t][1], centroids_gen[t][1]],
                    [centroids_true[t][2], centroids_gen[t][2]],
                    'k--', alpha=0.4
                )

        ax.set_xlabel('R')
        ax.set_ylabel('G')
        ax.set_zlabel('B')
        ax.set_title("RGB Centroid Distance per Cell Type (3D)")
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=9, ncol=2)
        plt.tight_layout()
        plt.show()

    avg_dist = np.mean(distances)
    #print("dist", avg_dist)
    score = kl_to_similarity(avg_dist, min_kl=0.0, max_kl=1)
    return score


def neighbor_kmeans_composition_matching_score(true_map, gen_map1, gen_map2, gen_map3,
                                               k=50, n_clusters=5, visualize=True):
    maps = [gen_map1, gen_map2, gen_map3]
    scores = []
    ratios = []
    yticklabels = [
        "B", "CD4+ T", "CD57+ Ent", "CD66+ Ent", "CD7+ Imm",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
        "Goblet", "ICC", "Lymphatic", "M1 Mac", "M2 Mac",
        "MUC1+ Ent", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
        "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
    ]

    map_true = true_map.squeeze(0)
    H, W = map_true.shape
    coords_true = np.array([(x, y) for y in range(H) for x in range(W)])
    feats_true = map_true.flatten().cpu().numpy()
    mask_true = feats_true != 0
    feats_true = feats_true[mask_true]
    coords_true = coords_true[mask_true]
    hist_true = compute_hist(feats_true, coords_true, k=k)
    km_true = MiniBatchKMeans(n_clusters=n_clusters, random_state=0).fit(hist_true)
    cluster_labels_true = km_true.labels_
    centers = km_true.cluster_centers_
    #print("centers", centers)
    ratio_true = compute_ratio(feats_true)
    ratios.append(ratio_true)
    #print("ratio_true", ratio_true)
    if visualize:
        plot_cluster_map(cluster_labels_true, coords_true, "True Map Clustered", H, W)
        log2_fc_heatmap_from_centroids1(
            niche_clusters=centers,
            tissue_values=ratio_true,
            cell_types_fixed_order=yticklabels,
            k_value=n_clusters,
            row_order=None
        )

    for i in range(3):
        map_gen = maps[i].squeeze(0)
        coords_gen = np.array([(x, y) for y in range(H) for x in range(W)])
        feats_gen = map_gen.flatten().cpu().numpy()
        mask_gen = feats_gen != 0
        feats_gen = feats_gen[mask_gen]
        coords_gen = coords_gen[mask_gen]
        #print("type_gen1", feats_gen.shape)
        #print("Number of non-zero points:", len(feats_gen))

        hist_gen = compute_hist(feats_gen, coords_gen, k=k)
        ratio_gen = compute_ratio(feats_gen)
        ratios.append(ratio_gen)
        km_gen = MiniBatchKMeans(n_clusters=n_clusters, random_state=0)
        cluster_labels_gen = km_gen.fit_predict(hist_gen)
        centers = km_gen.cluster_centers_
        comp_gen = compute_cluster_celltype_composition(cluster_labels_gen, feats_gen, n_clusters)

        if visualize:
            plot_cluster_map(cluster_labels_gen, coords_gen, f"Gen Map {i + 1} Clustered", H, W)
            df = pd.DataFrame(coords_gen, columns=["X", "Y"])
            df["Clusterid"] = cluster_labels_gen
            df["Exp"] = "1"
            catplot2(df, hue="Clusterid", exp="Exp", X="X", Y="Y", invert_y=True)

        row_order, col_order, cost_matrix, total_dist = compare_cluster_feature_centroids(
            hist_true, cluster_labels_true,
            hist_gen, cluster_labels_gen,
            n_clusters=n_clusters,
            metric='euclidean'
        )

        comp_gen_aligned = comp_gen[col_order]

        if visualize:
            log2_fc_heatmap_from_centroids1(
                niche_clusters=centers,
                tissue_values=ratio_gen,
                cell_types_fixed_order=yticklabels,
                k_value=n_clusters,
                row_order=range(comp_gen_aligned.shape[0])
            )

        scores.append(1/(1+total_dist))

    return scores


def cell_density_score_new(true_map, gen_map1, gen_map2, gen_map3,
                           k=20, bin_width=5.0, penalty_value=1.0,
                           alpha=0.5, visualize=True):
    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    from scipy.special import rel_entr  # for KL divergence
    import matplotlib.pyplot as plt

    def get_coords_and_feats(types):
        H, W = types.shape[1:]
        coords = np.array([(i, j) for i in range(H) for j in range(W)])
        feats = types.squeeze(0).flatten().cpu().numpy()
        mask = feats != 0
        return coords[mask], feats[mask]

    def get_distance_distributions(coords, feats):
        dists_by_type = [None] * 25
        for t in range(0, 25):
            points = coords[feats == t]
            if len(points) < k + 1:
                continue
            nbrs = NearestNeighbors(n_neighbors=k + 1).fit(points)
            dists, _ = nbrs.kneighbors(points)
            mean_dists = dists[:, 1:].mean(axis=1)
            dists_by_type[t] = mean_dists
        return dists_by_type

    maps = [true_map, gen_map1, gen_map2, gen_map3]
    all_dists_by_map = [get_distance_distributions(*get_coords_and_feats(m)) for m in maps]

    # Step 2: 为每个 cell type 使用 true map 的最大值作为 bin 边界
    true_dists_max = [np.max(d) if d is not None else 1.0 for d in all_dists_by_map[0]]
    bin_edges_list = [np.arange(0, max_v + bin_width, bin_width) for max_v in true_dists_max]

    num_gen = 3
    score_matrix = [[] for _ in range(num_gen)]

    for t in range(0, 25):
        all_dists = [d[t] for d in all_dists_by_map]
        if any(x is None for x in all_dists):
            for i in range(num_gen):
                score_matrix[i].append(1 / (1 + penalty_value))
            continue

        bin_edges = bin_edges_list[t]
        hists = []
        for d in all_dists:
            hist, _ = np.histogram(d, bins=bin_edges)
            hist = hist.astype(np.float32) + 1e-8
            hist /= hist.sum()
            hists.append(hist)

        true_hist = hists[0]
        for i in range(num_gen):
            kl = np.sum(rel_entr(true_hist, hists[i + 1]))  # non-normalized KL
            score = 1 / (1 + kl)
            score_matrix[i].append(score)

            if visualize:
                plt.figure(figsize=(6, 3))
                x = bin_edges[:-1]
                plt.plot(x, true_hist, label='True', linewidth=2)
                plt.plot(x, hists[i + 1], label=f'Gen{i+1}', linestyle='--')
                plt.title(f"Cell Type {t} Dist Histogram (True vs Gen{i+1})")
                plt.xlabel("Mean Distance to Neighbors")
                plt.ylabel("Frequency")
                plt.legend()
                plt.tight_layout()
                plt.show()

    type_scores = [np.mean(s) for s in score_matrix]

    def extract_all_distances(cell_map):
        coords = np.argwhere(cell_map.squeeze().cpu().numpy() != 0)
        if len(coords) <= k:
            return None
        nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
        dists, _ = nbrs.kneighbors(coords)
        return dists[:, 1:].mean(axis=1)

    true_dists = extract_all_distances(true_map)
    global_scores = []

    max_global = true_dists.max() if true_dists is not None else 1.0
    bin_edges_global = np.arange(0, max_global + bin_width, bin_width)

    for gen_map in [gen_map1, gen_map2, gen_map3]:
        gen_dists = extract_all_distances(gen_map)
        if true_dists is None or gen_dists is None:
            global_scores.append(1 / (1 + penalty_value))
            continue

        true_hist, _ = np.histogram(true_dists, bins=bin_edges_global)
        gen_hist, _ = np.histogram(gen_dists, bins=bin_edges_global)
        true_hist = true_hist.astype(np.float32) + 1e-8
        gen_hist = gen_hist.astype(np.float32) + 1e-8
        true_hist /= true_hist.sum()
        gen_hist /= gen_hist.sum()

        kl = np.sum(rel_entr(true_hist, gen_hist))
        score = 1 / (1 + kl)
        global_scores.append(score)

    final_scores = [alpha * type_scores[i] + (1 - alpha) * global_scores[i] for i in range(num_gen)]
    return final_scores

import lpips
lpips_loss_fn = lpips.LPIPS(net='vgg').to("cuda")

def spatial_structure_score(true_map, gen_map):
    """
    Compute LPIPS perceptual similarity between two RGB maps.
    Input: [3, H, W], float in [0, 1] (normalized image)
    Returns: float score, lower is more similar
    """
    m1 = torch.as_tensor(true_map).squeeze().to("cuda")
    m2 = torch.as_tensor(gen_map).squeeze().to("cuda")

    if m1.ndim != 3 or m1.shape[0] != 3:
        raise ValueError("Input map must have shape [3, H, W]")

    # Normalize from [0,1] to [-1,1] as required by LPIPS
    m1 = m1 * 2 - 1
    m2 = m2 * 2 - 1

    m1 = m1.unsqueeze(0)  # Add batch dimension: [1, 3, H, W]
    m2 = m2.unsqueeze(0)

    with torch.no_grad():
        score = lpips_loss_fn(m1, m2).item()

    return 1 - score


def cell_type_distribution(true_map, generated_map, show_plot=True, type_color_dict=None):
    type_color_dict = {
        1: [134.9, 124.1, 8.7],
        2: [62.3, 216.0, 132.0],
        3: [130.9, 46.4, 182.5],
        4: [92.8, 63.6, 202.5],
        5: [140.1, 239.7, 189.0],
        6: [200.0, 183.1, 168.9],
        7: [108.3, 16.1, 142.3],
        8: [132.7, 217.1, 223.4],
        9: [170.6, 96.0, 98.0],
        10: [68.4, 101.1, 101.3],
        11: [4.6, 146.9, 145.6],
        12: [177.1, 1.5, 112.2],
        13: [202.6, 136.4, 12.3],
        14: [95.1, 244.3, 209.6],
        15: [32.5, 214.2, 217.8],
        16: [176.2, 78.6, 193.2],
        17: [185.9, 206.0, 77.8],
        18: [212.0, 39.3, 35.0],
        19: [117.1, 190.4, 53.4],
        20: [101.2, 145.2, 246.9],
        21: [13.7, 179.1, 133.2],
        22: [247.4, 125.5, 111.5],
        23: [122.9, 151.0, 150.0],
        24: [118.0, 45.6, 47.7],
        25: [31.1, 114.7, 222.5]
    }
    """
    Compare the global cell type distribution between the true and generated maps,
    using KL divergence. Each bar is colored by cell type RGB; true uses solid bars, generated uses hatched.

    Args:
        true_map: torch.Tensor, shape [1, H, W]
        generated_map: torch.Tensor, shape [1, H, W]
        show_plot: bool
        type_color_dict: dict[int -> list of RGB in 0-255], key=cell type (1~25)

    Returns:
        similarity_score: float
    """
    import numpy as np
    import matplotlib.pyplot as plt

    cell_type_names = [
        "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte", "Goblet", "ICC",
        "Lymphatic", "M1 Macrophage", "M2 Macrophage", "MUC1+ Enterocyte", "NK",
        "Nerve", "Neuroendocrine", "Neutrophil", "Paneth", "Plasma",
        "Smooth muscle", "Stroma", "TA"
    ]

    true_map = true_map.squeeze().cpu().numpy()
    generated_map = generated_map.squeeze().cpu().numpy()

    unique_true = np.unique(true_map)
    unique_gen = np.unique(generated_map)
    all_labels = np.union1d(unique_true, unique_gen)
    all_labels = all_labels[all_labels != 0]

    if all_labels.size == 0:
        return 1.0

    num_classes = int(all_labels.max()) + 1
    true_hist, _ = np.histogram(true_map, bins=np.arange(1, num_classes + 1))
    gen_hist, _ = np.histogram(generated_map, bins=np.arange(1, num_classes + 1))

    if show_plot:
        # Normalize histograms to ratio
        true_ratio = true_hist / (true_hist.sum() + 1e-8)
        gen_ratio = gen_hist / (gen_hist.sum() + 1e-8)

        x = np.arange(1, num_classes)
        fig, ax = plt.subplots(figsize=(12, 4))

        default_color = (0.6, 0.6, 0.6)
        xticks = []
        for i in x:
            # bar color
            color = (np.array(type_color_dict[i]) / 255.0) if (
                        type_color_dict and i in type_color_dict) else default_color
            label = cell_type_names[i - 1] if 1 <= i <= len(cell_type_names) else f"Type {i}"

            # solid bar for true
            ax.bar(i - 0.18, true_ratio[i - 1], width=0.35, label=f'True-{label}' if i == 1 else "",
                   color=color, edgecolor='black')

            # hatched bar for generated
            ax.bar(i + 0.18, gen_ratio[i - 1], width=0.35, label=f'Generated-{label}' if i == 1 else "",
                   color=color, edgecolor='black', hatch='///', linewidth=0.8)

            xticks.append(label)

        ax.set_xticks(x)
        ax.set_xticklabels(xticks, rotation=45, ha='right')
        ax.set_xlabel("Cell Type")
        ax.set_ylabel("Proportion")
        ax.set_ylim(0, 1.05 * max(true_ratio.max(), gen_ratio.max()))
        ax.set_title("Cell Type Distribution (Ratio): True (solid) vs Generated (hatched)")
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(facecolor='gray', edgecolor='black', label='True'),
            Patch(facecolor='gray', edgecolor='black', hatch='///', label='Generated')
        ], loc='upper left')
        plt.tight_layout()
        plt.show()

    if true_hist.sum() == 0 or gen_hist.sum() == 0:
        return 0.0

    eps = 1e-8
    true_hist = (true_hist + eps) / (true_hist.sum() + eps * len(true_hist))
    gen_hist = (gen_hist + eps) / (gen_hist.sum() + eps * len(gen_hist))

    missing_mask = (true_hist > eps) & (gen_hist <= eps)
    kl_div = np.sum(true_hist * np.log(true_hist / (gen_hist + eps)))
    penalty_weight = 2

    missing_penalty = penalty_weight * np.sum(true_hist[missing_mask])
    #print(missing_penalty)
    kl_div += missing_penalty
    #print("KL divergence =", kl_div)

    similarity_score = 1 / (1 + kl_div*kl_div)
    return similarity_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.0 MB/s eta 0:00:00
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 78.9MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth


In [ ]:
#MAIN ENTRANCE
def evaluate_generated_maps(true_img_path, gen_img_path1, gen_img_path2, gen_img_path3, model_path):
    model = Autoencoder().to("cuda")
    model.load_state_dict(torch.load(model_path))
    model.eval()

    true_map = load_and_recover_z3d_png(true_img_path).to("cuda")
    gen_map1 = load_and_recover_z3d_png(gen_img_path1).to("cuda")
    gen_map2 = load_and_recover_z3d_png(gen_img_path2).to("cuda")
    gen_map3 = load_and_recover_z3d_png(gen_img_path3).to("cuda")

    type_true = infer_cell_map(true_map, model)
    type_gen1 = infer_cell_map(gen_map1, model)
    df_gen1 = save_inferred_map_to_df(type_gen1, region_id=1)
    type_gen2 = infer_cell_map(gen_map2, model)
    df_gen2 = save_inferred_map_to_df(type_gen2, region_id=1)
    type_gen3 = infer_cell_map(gen_map3, model)

    s1_1 = rgb_centroid_distance_score(true_map, gen_map1, type_true, type_gen1, visualize=False)
    s1_2 = rgb_centroid_distance_score(true_map, gen_map2, type_true, type_gen2, visualize=False)
    s1_3 = rgb_centroid_distance_score(true_map, gen_map3, type_true, type_gen3, visualize=False)
    s2 = neighbor_kmeans_composition_matching_score(type_true, type_gen1, type_gen2, type_gen3, k=10, n_clusters=20, visualize=False)
    s3 = cell_density_score_new(type_true, type_gen1, type_gen2, type_gen3, k=50, visualize=False)

    s4_1 = spatial_structure_score(true_map, gen_map1)
    s4_2 = spatial_structure_score(true_map, gen_map2)
    s4_3 = spatial_structure_score(true_map, gen_map3)

    s5_1 = cell_type_distribution(type_true, type_gen1, show_plot=False)
    s5_2 = cell_type_distribution(type_true, type_gen2, show_plot=False)
    s5_3 = cell_type_distribution(type_true, type_gen3, show_plot=False)

    return [s1_1, s1_2, s1_3], s2, s3, [s4_1, s4_2, s4_3], [s5_1, s5_2, s5_3]

umap_scores, kmeans_scores, density_scores, ss_scores, dist_scores = evaluate_generated_maps(
            f"1.png",###CHANGE THIS PATH AS ORIG IMAGE
            f"2.png",###CHANGE THIS PATH AS GENERATED IMAGE
            f"2.png",###CHANGE THIS PATH AS GENERATED IMAGE
            f"2.png",###CHANGE THIS PATH AS GENERATED IMAGE
            "drive/MyDrive/newae2.pth"
            )

/tmp/ipython-input-3179706075.py:83: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))


RuntimeError: shape '[-1, 3]' is invalid for input of size 262144

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

base_dir = "/content/drive/MyDrive/20251006decoded_eval/20251006Region9RecoveredNogeneration_SPLIT"
model_path = "/content/drive/MyDrive/newae2.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

cell_type_names = [
    "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
    "Goblet", "ICC", "Lymphatic", "M1 Macrophage", "M2 Macrophage",
    "MUC1+ Enterocyte", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
    "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
]


type_color_dict = {
    1: [134.9, 124.1, 8.7],
    2: [62.3, 216.0, 132.0],
    3: [130.9, 46.4, 182.5],
    4: [92.8, 63.6, 202.5],
    5: [140.1, 239.7, 189.0],
    6: [200.0, 183.1, 168.9],
    7: [108.3, 16.1, 142.3],
    8: [132.7, 217.1, 223.4],
    9: [170.6, 96.0, 98.0],
    10: [68.4, 101.1, 101.3],
    11: [4.6, 146.9, 145.6],
    12: [177.1, 1.5, 112.2],
    13: [202.6, 136.4, 12.3],
    14: [95.1, 244.3, 209.6],
    15: [32.5, 214.2, 217.8],
    16: [176.2, 78.6, 193.2],
    17: [185.9, 206.0, 77.8],
    18: [212.0, 39.3, 35.0],
    19: [117.1, 190.4, 53.4],
    20: [101.2, 145.2, 246.9],
    21: [13.7, 179.1, 133.2],
    22: [247.4, 125.5, 111.5],
    23: [122.9, 151.0, 150.0],
    24: [118.0, 45.6, 47.7],
    25: [31.1, 114.7, 222.5]
}

def compute_ratio_from_types(type_map: np.ndarray, num_types: int = 25) -> np.ndarray:
    a = type_map.astype(int)
    mask = (a >= 1) & (a <= num_types)
    if not np.any(mask):
        return np.zeros(num_types, dtype=float)
    a_fg = a[mask] - 1  # 0-based
    counts = np.bincount(a_fg.ravel(), minlength=num_types)
    ratio = counts / counts.sum()
    return ratio

@torch.no_grad()
def infer_distribution(img_path: str, model: torch.nn.Module, device: str = "cuda") -> np.ndarray:
    z3d = load_and_recover_z3d_png(img_path).to(device)
    pred_types = infer_cell_map(z3d, model)
    if isinstance(pred_types, torch.Tensor):
        pred_types = pred_types.squeeze().detach().cpu().numpy()
    return compute_ratio_from_types(pred_types, num_types=25)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def main():

    order_types = list(type_color_dict.keys())                           # e.g., [1,2,...,25] 但以字典写入顺序为准
    idx = np.array([t - 1 for t in order_types])                         # 0-based
    cell_type_names_ord = [cell_type_names[t - 1] for t in order_types]
    color_list_ord = [np.array(type_color_dict[t]) / 255.0 for t in order_types]

    model = Autoencoder().to(device)
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()


    tissue_order = ["L"] + [str(i) for i in range(8)] + ["R"]
    path_map = {name: os.path.join(base_dir, f"{name}.png") for name in tissue_order}


    missing = [k for k, p in path_map.items() if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(f"以下文件未找到：{missing}. 请检查 base_dir = {base_dir}")


    distributions = []
    for name in tissue_order:
        p = path_map[name]
        ratio = infer_distribution(p, model, device=device)
        distributions.append(ratio)
    distributions = np.stack(distributions, axis=0)  # (10, 25)


    distributions_ord = distributions[:, idx]         # (10, 25)


    x = np.arange(len(tissue_order))                  # 0..9
    y_values = distributions_ord.T                    # (25, 10)

    fig1, ax1 = plt.subplots(figsize=(16, 6))
    ax1.stackplot(
        x,
        *y_values,
        labels=cell_type_names_ord,
        colors=color_list_ord,
        alpha=0.9
    )
    ax1.set_xticks(x)
    ax1.set_xticklabels(tissue_order, rotation=0)
    ax1.set_ylim(0, 1.0)
    ax1.set_ylabel("Proportion")
    ax1.set_title("Cell Type Distribution Wave (L, 0–7, R)")

    leg1 = ax1.legend(
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        fontsize=7,
        frameon=False,
        ncol=1
    )
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    ensure_dir(base_dir)
    wave_path = os.path.join(base_dir, "distribution_wave.png")
    plt.savefig(wave_path, dpi=300)
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(16, 6))
    bottoms = np.zeros(len(tissue_order), dtype=float)

    for i, (name_i, color_i) in enumerate(zip(cell_type_names_ord, color_list_ord)):
        vals = distributions_ord[:, i]  # (10,)
        ax2.bar(
            x,
            vals,
            bottom=bottoms,
            color=color_i,
            edgecolor='none',
            label=name_i,
            width=0.85
        )
        bottoms += vals

    ax2.set_xticks(x)
    ax2.set_xticklabels(tissue_order, rotation=0)
    ax2.set_ylim(0, 1.0)
    ax2.set_ylabel("Proportion")
    ax2.set_title("Cell Type Distribution (Stacked Bars) for L, 0–7, R")
    ax2.grid(axis='y', linestyle='--', alpha=0.25)

    leg2 = ax2.legend(
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        fontsize=7,
        frameon=False,
        ncol=1
    )
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    bars_path = os.path.join(base_dir, "distribution_stacked_bars.png")
    plt.savefig(bars_path, dpi=300)
    plt.show()

    print(f"Saved:\n  {wave_path}\n  {bars_path}")


In [ ]:

if __name__ == "__main__":
    main()
